In [1]:
#Milestone 4: A/B Test Design for top priority feature
#Top Feature from rice_scores.csv: "Advance Nutrition/Meal Planning
#Rice Score: 10000.0 (Rank 1)
#Current MAU: 50,000 users
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
print("="*60)
print("Step 1: LOADING PREVIOUS MILESTONE DATA TO CONFIRM TOP FEATURE")
print("="*60)

rice_df = pd.read_csv('rice_scores.csv')
top5_df = pd.read_csv('top5_features.csv')

print("Top 5 features from M3:")
print(top5_df.head())

top_feature = rice_df.loc[rice_df['RICE_Score'].idxmax(), 'Feature']
print(f"\nSelected feature for A/B test: {top_feature}")
print("RICE_Score:", rice_df['RICE_Score'].max())

Step 1: LOADING PREVIOUS MILESTONE DATA TO CONFIRM TOP FEATURE
Top 5 features from M3:
                                     Feature  Reach  Impact  Confidence  \
0           Advanced Nutrition/Meal Planning  20000     2.0         1.0   
1                               Offline Mode   9000     3.0         1.0   
2                    Rest Timer Between Sets   3500     2.0         1.0   
3              Video Exercise Demonstrations   3000     2.0         0.8   
4  Wearable Integration (Apple Watch/Fitbit)   5000     2.0         0.8   

   Effort  Avg_Sentiment  Est_Mentions  RICE_Score  Rank  
0       4          0.609        839.80     10000.0     1  
1       3          0.806         10.20      9000.0     2  
2       1          0.557        546.38      7000.0     3  
3       2          0.557        546.38      2400.0     4  
4       4          0.471        293.42      2000.0     5  

Selected feature for A/B test: Advanced Nutrition/Meal Planning
RICE_Score: 10000.0


In [3]:
print("="*60)
print("Step 2: DEFINING A/B TEST PARAMETERS")
print("="*60)

#Defining A/B test parameters
feature_name = "Advanced Nutrition/Meal Planning"

#Hypotheses
null_hypothesis = "Adding Advanced Nutrition/Meal Planning will NOT increase day-7 retention rate."
alt_hypothesis = "Adding Advanced Nutrition/Meal Planning will increase day-7 retention rate by atleast 5 percentage points."

print("\nHypotheses:")
print("Null (H0):", null_hypothesis)
print("Alternative (H1):", alt_hypothesis)

#Test groups 
group_a_size_pct = 0.5 #Contril: 50%
group_b_size_pct = 0.5 #Treatment: 50%

Step 2: DEFINING A/B TEST PARAMETERS

Hypotheses:
Null (H0): Adding Advanced Nutrition/Meal Planning will NOT increase day-7 retention rate.
Alternative (H1): Adding Advanced Nutrition/Meal Planning will increase day-7 retention rate by atleast 5 percentage points.


In [4]:
print("="*60)
print("Step 3: DEFINING SUCCESS METRICS")
print("="*60)

primary_metrics = "Day-7 rentention_rate"
secondary_metrics = [
    "Session frequency (sessions/week)",
    "Feature engagement rate (% of users using feature)",
    "App rating improvement"
]
guardrail_metrics = [
    "App performance (load time)",
    "Crash-free session rate",
    "Uninstall rate"
]

print("\nSucess Metrics:")
print("Primary:", primary_metrics)
print("Secondary:", secondary_metrics)
print("Guardrail:", guardrail_metrics)

Step 3: DEFINING SUCCESS METRICS

Sucess Metrics:
Primary: Day-7 rentention_rate
Secondary: ['Session frequency (sessions/week)', 'Feature engagement rate (% of users using feature)', 'App rating improvement']
Guardrail: ['App performance (load time)', 'Crash-free session rate', 'Uninstall rate']


In [5]:
print("="*60)
print("Step 4: SAMPLE SIZE CALCULATION")
print("="*60)

baseline_retention = 0.22 #current Day-7 retention: 22%
mde = 0.05 #minimum detectable effect: 5% points (to 27%)
confidence_level = 0.95 #95% confidence
power = 0.80 #80% power
alpha = 1 - confidence_level 
z_alpha = stats.norm.ppf(1 - alpha / 2) #1.96
z_beta = stats.norm.ppf(power) #0.8416

#sample size per group for two-proprotion z-test
p_pooled = baseline_retention #conservaative approximation
n_per_group = (2 * (z_alpha + z_beta)** 2 * p_pooled * (1 - p_pooled)) / mde**2
total_sample_size = 2 * n_per_group

print("\nSample size calculation:")
print(f"Baseline retention (p): {baseline_retention}")
print(f"MDE: {mde}")
print(f"Z_alpha/2: {z_alpha:.2f}")
print(f"Z_beta: {z_beta:.2f}")
print(f"Sample size per group: {np.ceil(n_per_group):.0f}")
print(f"Total sample size: {np.ceil(total_sample_size):.0f} users")

Step 4: SAMPLE SIZE CALCULATION

Sample size calculation:
Baseline retention (p): 0.22
MDE: 0.05
Z_alpha/2: 1.96
Z_beta: 0.84
Sample size per group: 1078
Total sample size: 2155 users


In [6]:
print("="*60)
print("Step 5: TEST DURATION ESTIMATION")
print("="*60)

mau = 50000 #Monthly Active Users 
new_users_per_month = 10000 #From project assumptions
days_to_reach_sample = (total_sample_size / new_users_per_month) * 30
test_duration_days = days_to_reach_sample + 7 # +7 days for day-7 retention
print(f"\nTest Duration:")
print(f"Estimated days to reach simple: {days_to_reach_sample:.1f}")
print(f"Minimum test duration: {np.ceil(test_duration_days):.0f} days")
print("Recommendation: Run for 28 days to capture weekly patterns.")

Step 5: TEST DURATION ESTIMATION

Test Duration:
Estimated days to reach simple: 6.5
Minimum test duration: 14 days
Recommendation: Run for 28 days to capture weekly patterns.


In [7]:
print("="*60)
print("Step 6: STATISTICAL TEST SIMULATION (POWER ANALYSIS EXAMPLE)")
print("="*60)

#Simulating data for illustration 
np.random.seed(42)
n = int(n_per_group)

#control group : baseline 22% 
control_retention = np.random.binomial (1, baseline_retention, n).mean()

#treatment group : improved 27%
treatment_retention = np.random.binomial (1, baseline_retention + mde, n). mean()

#Two proportion z-test
count_a = np.sum(np.random.binomial(1, baseline_retention, n))
count_b = np.sum(np.random.binomial(1, baseline_retention + mde, n))
n_a, n_b = n, n 

p_a = count_a / n_a 
p_b = count_b / n_b
p_pooled = (count_a + count_b) / (n_a + n_b)
se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_a + 1 / n_b))
z_stat = (p_b - p_a) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("\nStatistical Test Simulation (example with calculated sample size):")
print(f"Control retention: {control_retention:.3f}")
print(f"Treatment retention: {treatment_retention:.3f}")
print(f"Z-statistic: {z_stat:.3f}")
print(f"P-vaues: {p_value:.4f}")
if p_value < 0.05:
    print( "Results: Reject H0 - Significant improvement detected.")
else: 
    print("Result: Fail to reject H0.")

#Decision rule
print("\nGo/No-Go criteria:")
print("- If p-value < 0.05 AND retention lift >= 5%: Launch to 100% users.")
print("- if p-values >= 0.05 but list >= 5%: iterate on feature.")
print("- if p-values >= 0.05: De-prioritize feature.")

Step 6: STATISTICAL TEST SIMULATION (POWER ANALYSIS EXAMPLE)

Statistical Test Simulation (example with calculated sample size):
Control retention: 0.222
Treatment retention: 0.276
Z-statistic: 2.187
P-vaues: 0.0287
Results: Reject H0 - Significant improvement detected.

Go/No-Go criteria:
- If p-value < 0.05 AND retention lift >= 5%: Launch to 100% users.
- if p-values >= 0.05 but list >= 5%: iterate on feature.
- if p-values >= 0.05: De-prioritize feature.


In [8]:
print("="*60)
print("Step 7: IMPLEMENTATION PLAN TABLE")
print("="*60)

plan_data = {
        'Month': ['1', '2-3', '4'],
    'Phase': ['Pre-Launch', 'Test Running', 'Analysis'],
    'Tasks': [
        'Develop feature, setup tracking, QA testing.',
        'Launch to 50% new users, monitor metrics daily.',
        'Run stats tests, decide go/no-go.'
    ]
}
plan_df = pd.DataFrame(plan_data)
print("\nImplementation Plan:")
print(plan_df.to_string(index=False))

Step 7: IMPLEMENTATION PLAN TABLE

Implementation Plan:
Month        Phase                                           Tasks
    1   Pre-Launch    Develop feature, setup tracking, QA testing.
  2-3 Test Running Launch to 50% new users, monitor metrics daily.
    4     Analysis               Run stats tests, decide go/no-go.


In [9]:
print("="*60)
print("Step 8: RISK MITIGATION TABLE")
print("="*60)

risks_data = {
     'Risk': [
        'Feature causes crashes.',
        'Negative user feedback.',
        'Low feature engagement.',
        'Seasonality effects.'
    ],
    'Mitigation': [
        'Start 10% rollout, monitor crashes, rollback if >2%.',
        'Monitor reviews daily, disable remotely if needed.',
        'Add in-app tutorial, track discovery rate.',
        'Run full month, compare to historical data.'
    ]
}
risks_df = pd.DataFrame(risks_data)
print("\nRisk Mitigation:")
print(risks_df.to_string(index=False))

Step 8: RISK MITIGATION TABLE

Risk Mitigation:
                   Risk                                           Mitigation
Feature causes crashes. Start 10% rollout, monitor crashes, rollback if >2%.
Negative user feedback.   Monitor reviews daily, disable remotely if needed.
Low feature engagement.           Add in-app tutorial, track discovery rate.
   Seasonality effects.          Run full month, compare to historical data.


In [10]:
# Step 9: Save A/B Test Summary CSV for dashboard use
ab_summary = {
    'Feature': [feature_name],
    'RICE_Score': [rice_df['RICE_Score'].max()],
    'Primary_Metric': [primary_metrics],
    'Baseline': [baseline_retention],
    'MDE': [mde],
    'Sample_Per_Group': [np.ceil(n_per_group)],
    'Total_Sample': [np.ceil(total_sample_size)],
    'Min_Duration_Days': [np.ceil(test_duration_days)],
    'Null_Hypothesis': [null_hypothesis],
    'Alt_Hypothesis': [alt_hypothesis]
}
ab_df = pd.DataFrame(ab_summary)
ab_df.to_csv('ab_test_design.csv', index=False)
print("\nA/B Test Summary saved to 'ab_test_design.csv' for dashboard.")


A/B Test Summary saved to 'ab_test_design.csv' for dashboard.
